# A/B Test Analysis - `new_checkout`

This notebook is a deeper dive behind the pipeline artifact `outputs/ab_test_decision.md`.
It pulls data from Postgres and calculates:
- checkout→purchase conversion (24h)
- uplift (treatment - control)
- 95% confidence interval (normal approximation)
- guardrail: average order value (AOV)


In [ ]:
import os
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt

host = os.getenv('POSTGRES_HOST','localhost')
port = int(os.getenv('POSTGRES_PORT','5432'))
db = os.getenv('POSTGRES_DB','analytics')
user = os.getenv('POSTGRES_USER','analytics')
pw = os.getenv('POSTGRES_PASSWORD','analytics')

conn = psycopg2.connect(host=host, port=port, dbname=db, user=user, password=pw)
conn.autocommit = True
print('Connected')


In [ ]:
# Use dbt mart if available
q = """
select event_day::date as day, variant, users_started_checkout, users_purchased_24h,
       checkout_to_purchase_rate_24h as rate
from analytics.mart_conversion_daily
where event_day >= now() - interval '30 days'
order by 1,2;
"""
df = pd.read_sql(q, conn)
df.head()


In [ ]:
# Aggregate overall experiment result (over full window)
agg = df.groupby('variant')[['users_started_checkout','users_purchased_24h']].sum()
agg['rate'] = agg['users_purchased_24h'] / agg['users_started_checkout']
agg


In [ ]:
import math

c = agg.loc['control']
t = agg.loc['treatment']

p1 = c['rate']; n1 = c['users_started_checkout']
p2 = t['rate']; n2 = t['users_started_checkout']
diff = p2 - p1

se = math.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci_low = diff - 1.96*se
ci_high = diff + 1.96*se

print(f"control={p1:.4f} (n={int(n1):,})")
print(f"treat  ={p2:.4f} (n={int(n2):,})")
print(f"uplift ={diff:.4f} 95% CI [{ci_low:.4f}, {ci_high:.4f}]")


In [ ]:
# Guardrail: AOV by variant (from fct_orders)
q_aov = """
select variant, avg(revenue) as aov, count(*) as orders
from analytics.fct_orders
where variant in ('control','treatment')
group by 1
order by 1;
"""
aov = pd.read_sql(q_aov, conn)
aov


In [ ]:
# Plot daily conversion rates
pivot = df.pivot(index='day', columns='variant', values='rate')
pivot.plot(figsize=(10,4))
plt.xlabel('day')
plt.ylabel('checkout→purchase rate (24h)')
plt.tight_layout()
plt.show()
